In [1]:
import pandas as pd
import numpy as np


## Q1: Load Dataset and Analyze Sales by Region

In this task, we load the Superstore dataset, create a Series of total sales per Order ID, and analyze the distribution of sales across different regions using descriptive statistics and index-based selection.

In [2]:
df = pd.read_csv("D:\EDAV Assignment\Sample-Superstore.csv",encoding='latin1')

sales_per_order = df.groupby('Order ID')['Sales'].sum()

region_stats = df.groupby('Region')['Sales'].describe()

east_sales = region_stats.loc['East']

print(region_stats)
print("\nEast Region Stats:\n", east_sales)

          count        mean         std    min     25%     50%       75%  \
Region                                                                     
Central  2323.0  215.772661  632.779010  0.444  14.620  45.980  200.0120   
East     2848.0  238.336110  620.712652  0.852  17.520  54.900  209.6175   
South    1620.0  241.803645  774.796273  1.167  17.187  54.594  208.7220   
West     3203.0  226.493233  524.876877  0.990  19.440  60.840  215.8090   

               max  
Region              
Central  17499.950  
East     11199.968  
South    22638.480  
West     13999.960  

East Region Stats:
 count     2848.000000
mean       238.336110
std        620.712652
min          0.852000
25%         17.520000
50%         54.900000
75%        209.617500
max      11199.968000
Name: East, dtype: float64


### Analysis

From the above results, we can see that sales are different for each region. Some regions have higher average sales while others have lower values. By selecting a specific region like East using indexing, we can study its sales distribution separately. This helps in understanding which region is performing better.

## Q2: Profit Margin Analysis

We calculate profit margin as Profit/Sales and identify categories and sub-categories with highest and lowest performance.

In [3]:
df['Profit_Margin'] = df['Profit'] / df['Sales']

cat_margin = df.groupby('Category')['Profit_Margin'].mean()
subcat_margin = df.groupby('Sub-Category')['Profit_Margin'].mean()

print("Top Category:\n", cat_margin.idxmax(), cat_margin.max())
print("Worst Category:\n", cat_margin.idxmin(), cat_margin.min())

print("\nWorst Subcategories:\n", subcat_margin.nsmallest(5))

Top Category:
 Technology 0.15613805312776619
Worst Category:
 Furniture 0.0387835332152663

Worst Subcategories:
 Sub-Category
Binders      -0.199595
Appliances   -0.156869
Tables       -0.147727
Bookcases    -0.126640
Machines     -0.072026
Name: Profit_Margin, dtype: float64


### Analysis

After calculating the profit margin, we observe that all categories do not give the same profit. Some categories have higher margins while some are very low or even negative. The sub-categories with low values are underperforming and may need improvement.

## Q3: Hierarchical Indexing on Region and Category

We use multi-level indexing to analyze sales and profit across regions and categories.

In [5]:
multi = df.set_index(['Region', 'Category'])

agg_data = multi.groupby(level=[0,1])[['Sales','Profit']].sum()

west_data = agg_data.xs('West')

print(agg_data)
print("\nWest Region:\n", west_data)

                               Sales      Profit
Region  Category                                
Central Furniture        163797.1638  -2871.0494
        Office Supplies  167026.4150   8879.9799
        Technology       170416.3120  33697.4320
East    Furniture        208291.2040   3046.1658
        Office Supplies  205516.0550  41014.5791
        Technology       264973.9810  47462.0351
South   Furniture        117298.6840   6771.2061
        Office Supplies  125651.3130  19986.3928
        Technology       148771.9080  19991.8314
West    Furniture        252612.7435  11504.9503
        Office Supplies  220853.2490  52609.8490
        Technology       251991.8320  44303.6496

West Region:
                        Sales      Profit
Category                                
Furniture        252612.7435  11504.9503
Office Supplies  220853.2490  52609.8490
Technology       251991.8320  44303.6496


### Analysis

Using hierarchical indexing with Region and Category makes it easier to analyze the data at different levels. We can clearly see which category is doing well in which region. This helps in identifying strong and weak areas in business performance.

## Q4: Profitability Score

We normalize Profit, Discount, and Quantity and combine them into a single profitability score.

In [6]:
import numpy as np

df['Norm_Profit'] = (df['Profit'] - df['Profit'].min()) / (df['Profit'].max() - df['Profit'].min())
df['Norm_Discount'] = (df['Discount'] - df['Discount'].min()) / (df['Discount'].max() - df['Discount'].min())
df['Norm_Quantity'] = (df['Quantity'] - df['Quantity'].min()) / (df['Quantity'].max() - df['Quantity'].min())

df['Profitability_Score'] = df['Norm_Profit'] - df['Norm_Discount'] + df['Norm_Quantity']

top_products = df.sort_values(by='Profitability_Score', ascending=False).head(10)

print(top_products[['Product Name','Profitability_Score']])

                                           Product Name  Profitability_Score
9039   GBC Ibimaster 500 Manual ProClick Binding System             1.692836
1711  Acco 7-Outlet Masterpiece Power Center, Wihtou...             1.474042
7387  Electrix Architect's Clamp-On Swing Arm Lamp, ...             1.465838
1429  Avery 4027 File Folder Labels for Dot Matrix P...             1.453107
1433                  High-Back Leather Manager's Chair             1.450919
9168                                         Xerox 1964             1.449806
6628                  PureGear Roll-On Screen Protector             1.448955
9941  Memorex Mini Travel Drive 16 GB USB 2.0 Flash ...             1.445813
575         Personal Creations Ink Jet Cards and Labels             1.445250
3902  Eldon ProFile File 'N Store Portable File Tub ...             1.445046


### Analysis

The profitability score combines profit, discount, and quantity into one value. Products with high profit and quantity but low discount get better scores. This helps in finding top-performing products easily.

## Q5: Null Value Handling

We compare the impact of dropping null values vs imputing with group means.

In [7]:
nulls = df.isnull().sum()
print(nulls)

drop_df = df.dropna()
drop_revenue = drop_df['Sales'].sum()

df_impute = df.copy()
df_impute['Sales'] = df_impute.groupby('Region')['Sales'].transform(lambda x: x.fillna(x.mean()))
impute_revenue = df_impute['Sales'].sum()

print("Revenue after drop:", drop_revenue)
print("Revenue after impute:", impute_revenue)

Row ID                 0
Order ID               0
Order Date             0
Ship Date              0
Ship Mode              0
Customer ID            0
Customer Name          0
Segment                0
Country                0
City                   0
State                  0
Postal Code            0
Region                 0
Product ID             0
Category               0
Sub-Category           0
Product Name           0
Sales                  0
Quantity               0
Discount               0
Profit                 0
Profit_Margin          0
Norm_Profit            0
Norm_Discount          0
Norm_Quantity          0
Profitability_Score    0
dtype: int64
Revenue after drop: 2297200.8603
Revenue after impute: 2297200.8603


### Analysis

When we remove null values, some data is lost which reduces the total revenue. But when we fill missing values using mean, most of the data is kept. So, imputing values is better in this case compared to dropping rows.

## Q6: High Discount Loss Analysis

We filter orders with high discount and negative profit.

In [8]:
loss_df = df[(df['Discount'] > 0.3) & (df['Profit'] < 0)]

loss_subcat = loss_df.groupby('Sub-Category')['Profit'].mean()
overall_subcat = df.groupby('Sub-Category')['Profit'].mean()

comparison = pd.DataFrame({
    'Loss Data': loss_subcat,
    'Overall': overall_subcat
})

print(comparison.sort_values(by='Loss Data'))

               Loss Data     Overall
Sub-Category                        
Machines     -699.981353   29.432669
Tables       -223.736846  -55.565771
Bookcases    -175.698147  -15.230509
Appliances   -128.800615   38.922758
Phones        -69.234845   50.073938
Binders       -62.822996   19.843574
Furnishings   -43.077212   13.645918
Accessories          NaN   54.111788
Art                  NaN    8.200737
Chairs               NaN   43.095894
Copiers              NaN  817.909190
Envelopes            NaN   27.418019
Fasteners            NaN    4.375660
Labels               NaN   15.236962
Paper                NaN   24.856620
Storage              NaN   25.152277
Supplies             NaN   -6.258418


### Analysis

From the filtered data, we can see that orders with high discount and negative profit are causing losses. Some sub-categories are more affected than others. This shows that giving too much discount can reduce profit.

## Q7: Shipping Delay Classification

We classify orders as On-Time or Delayed based on shipping duration.

In [9]:
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

def shipping_status(row):
    return 'Delayed' if (row['Ship Date'] - row['Order Date']).days > 3 else 'On-Time'

df['Shipping Status'] = df.apply(shipping_status, axis=1)

delay_summary = df.groupby('Region')['Shipping Status'].value_counts()

print(delay_summary)

Region   Shipping Status
Central  Delayed            1659
         On-Time             664
East     Delayed            1891
         On-Time             957
South    Delayed            1097
         On-Time             523
West     Delayed            2120
         On-Time            1083
Name: count, dtype: int64


### Analysis

By checking order date and ship date, we classified orders as on-time or delayed. Some regions have more delayed orders, which may indicate problems in shipping or delivery process.

## Q8: Index Alignment Behavior

We observe how merging Series with mismatched indices introduces NaN values.

In [10]:
sales_series = df.groupby('Region')['Sales'].sum()
profit_series = df.groupby('Category')['Profit'].sum()

combined = pd.concat([sales_series, profit_series], axis=1)

print(combined)

                       Sales       Profit
Central          501239.8908          NaN
East             678781.2400          NaN
South            391721.9050          NaN
West             725457.8245          NaN
Furniture                NaN   18451.2728
Office Supplies          NaN  122490.8008
Technology               NaN  145454.9481


### Analysis

When we combine two Series with different indices, Pandas matches them using index labels. If the labels do not match, NaN values appear. This shows how index alignment works in Pandas.

## Q9: Segment and Category Analysis

We analyze average metrics and reshape using unstack().

In [11]:
seg_cat = df.groupby(['Segment','Category'])[['Sales','Profit','Quantity']].mean()

matrix = seg_cat.unstack()

print(matrix)

                  Sales                                 Profit  \
Category      Furniture Office Supplies  Technology  Furniture   
Segment                                                          
Consumer     351.347091      116.390194  427.339534   6.281293   
Corporate    354.519792      126.745309  444.855810  11.741201   
Home Office  336.825131      115.309021  535.976658  10.705465   

                                        Quantity                             
Category    Office Supplies Technology Furniture Office Supplies Technology  
Segment                                                                      
Consumer          18.014174  74.445646  3.743037        3.760154   3.782334  
Corporate         22.102923  79.723823  3.862229        3.856044   3.781588  
Home Office       24.034439  89.152458  3.776243        3.827618   3.646199  


### Analysis

Using Segment and Category together helps us understand average sales, profit, and quantity. After using unstack(), the data becomes easier to read and compare.

## Q10: Comprehensive Sales Analytics Report

This report includes:
- Null value audit
- Profitability KPIs
- Regional and category breakdown
- Loss analysis
- Shipping performance

In [12]:
print("Total Revenue:", df['Sales'].sum())
print("Total Profit:", df['Profit'].sum())
print("Top Region:", df.groupby('Region')['Sales'].sum().idxmax())
print("Worst Subcategory:", df.groupby('Sub-Category')['Profit'].sum().idxmin())

Total Revenue: 2297200.8603
Total Profit: 286397.0217
Top Region: West
Worst Subcategory: Tables


### Analysis

The final report gives overall information about sales, profit, and performance. It helps in identifying top regions, weak sub-categories, and overall business trends.